<a href="https://colab.research.google.com/github/sitnikovdev/Hands-On-Large-Language-Models/blob/main/Copy_of_Chapter_2_Tokens_and_Token_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Chapter 2 - Tokens and Token Embeddings</h1>
<i>Exploring tokens and embeddings as an integral part of building LLMs</i>


<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter02/Chapter%202%20-%20Tokens%20and%20Token%20Embeddings.ipynb)

---

This notebook is for Chapter 2 of the [Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961) book by [Jay Alammar](https://www.linkedin.com/in/jalammar) and [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/).

---

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961">
<img src="https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png" width="350"/></a>


### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [2]:
import os

# Папка для кэшированных пакетов на Drive
PACKAGES_PATH = '/content/drive/MyDrive/colab_packages'
os.makedirs(PACKAGES_PATH, exist_ok=True)

# Устанавливаем пакеты в эту


In [6]:
!rm -rf /content/drive/MyDrive/colab_packages

In [1]:
%%capture
!pip install --target={PACKAGES_PATH} --upgrade transformers==4.41.2 sentence-transformers==3.0.1 gensim==4.3.2 scikit-learn==1.5.0 accelerate==0.31.0 peft==0.11.1 scipy==1.10.1 numpy==1.26.


# В следующей сессии — только монтирование и  sys.path
В новом ноутбуке или после рестарта достаточно выполнить:
from google.colab import drive
drive.mount('/content/drive')


In [3]:
import sys
PACKAGES_PATH = '/content/drive/MyDrive/colab_packages'
if PACKAGES_PATH not in sys.path:
    sys.path.insert(0, PACKAGES_PATH)


# Пакеты уже доступны
import pandas as pd

Повторно устанавливать их не нужно, если версия Python и ABI совместимы.

In [ ]:
# %%capture
# !pip install --upgrade transformers==4.41.2 sentence-transformers==3.0.1 gensim==4.3.2 scikit-learn==1.5.0 accelerate==0.31.0 peft==0.11.1 scipy==1.10.1 numpy==1.26.4

In [4]:
import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/huggingface_cache'

# Downloading and Running An LLM

The first step is to load our model onto the GPU for faster inference. Note that we load the model and tokenizer separately and keep them as such so that we can explore them separately.

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import transformers
transformers.logging.set_verbosity_error()

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

In [6]:
embedding_matrix = model.get_input_embeddings().weight
print(embedding_matrix.shape)

torch.Size([32064, 3072])


In [7]:
embedding_matrix = model.get_input_embeddings().weight
print(embedding_matrix.shape)  # torch.Size([32064, 3072])

# Эмбеддинг конкретного токена
token_id = tokenizer("cat", add_special_tokens=False).input_ids[0]
vector = embedding_matrix[token_id]
print(vector.shape)  # torch.Size([3072])
print(vector[:10])   # первые 10 чисел вектора

torch.Size([32064, 3072])
torch.Size([3072])
tensor([ 0.0527, -0.0249,  0.0129, -0.0518,  0.0046, -0.0359, -0.0190,  0.0103,
        -0.0135, -0.0164], device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<SliceBackward0>)


Логичное продолжение раздела про токенизацию — раз токены у нас теперь есть (числа-идентификаторы), встаёт следующий вопрос: как модель вообще "понимает" эти числа? Разберём.


## Проблема, которую решают эмбеддинги

Токенайзер выдаёт **ID** — просто целые числа (мы это видели весь предыдущий разговор: `input_ids`, `token_ids`). Но число `1345` само по себе не несёт никакого смысла для нейросети — это произвольный индекс в словаре, как номер страницы в словаре. Если бы модель работала прямо с этими числами, она бы решила, что токен `1346` "ближе по смыслу" к `1345`, чем к `50000` — просто потому что числа рядом, что совершенно бессмысленно (соседние ID в словаре — это часто случайно оказавшиеся рядом по частоте слова, не по смыслу).

**Token embedding** — это способ заменить "голый номер" на **вектор** (обычно от пары сотен до нескольких тысяч чисел с плавающей точкой), который *кодирует смысл* токена в многомерном пространстве.

## Embedding-матрица — по сути огромная таблица поиска

Технически это одна большая матрица размером `vocab_size × embedding_dimension`. Помнишь из GGUF-лога, который мы разбирали:

```
phi3.embedding_length = 3072
n_vocab = 32064
```

Значит у Phi-3 это матрица размером **32064 × 3072** — для каждого из 32064 токенов словаря есть свой уникальный вектор из 3072 чисел. Получение эмбеддинга токена — это буквально **обращение по индексу** в эту таблицу: `embedding_matrix[token_id]`. Никаких сложных вычислений на этом этапе, просто lookup.

## Откуда берутся сами значения — и почему это не случайные числа

В самом начале обучения (до всякого тренинга) эта матрица инициализируется случайными числами — на этом этапе токены `"кот"` и `"космос"` находятся в векторном пространстве совершенно произвольно относительно друг друга.

В процессе обучения модели (предсказание следующего токена на огромном корпусе текста) эти векторы **постепенно перестраиваются** через backpropagation так, чтобы токены, встречающиеся в похожих контекстах, оказывались близко друг к другу в этом многомерном пространстве. Знаменитый пример, который часто приводит Аламмар в своих иллюстрациях (и который наверняка появится в этом разделе книги): после обучения векторное расстояние можно складывать и вычитать почти как арифметику — `king - man + woman ≈ queen`. Это не запрограммировано явно — это эмерджентное свойство, возникающее просто из статистики совместной встречаемости слов в тексте.



## Практическая демонстрация — можно посмотреть эмбеддинги прямо из уже загруженной модели

Раз у тебя Phi-3 уже загружена через `transformers` в предыдущих главах, эту таблицу можно достать напрямую:


In [ ]:

embedding_matrix = model.get_input_embeddings().weight
print(embedding_matrix.shape)  # torch.Size([32064, 3072])

# Эмбеддинг конкретного токена
token_id = tokenizer("cat", add_special_tokens=False).input_ids[0]
vector = embedding_matrix[token_id]
print(vector.shape)  # torch.Size([3072])
print(vector[:10])   # первые 10 чисел вектора


torch.Size([32064, 3072])
torch.Size([3072])
tensor([ 0.0527, -0.0249,  0.0129, -0.0518,  0.0046, -0.0359, -0.0190,  0.0103,
        -0.0135, -0.0164], device='cuda:0', dtype=torch.bfloat16,
       grad_fn=<SliceBackward0>)



## Косинусное сходство — как измерить "близость смысла"

Классический пример для раздела — сравнить несколько слов по сходству их векторов:



In [ ]:

import torch.nn.functional as F

def get_embedding(word):
    token_id = tokenizer(word, add_special_tokens=False).input_ids[0]
    return embedding_matrix[token_id]

cat = get_embedding("cat")
dog = get_embedding("dog")
car = get_embedding("car")

print("cat-dog:", F.cosine_similarity(cat.unsqueeze(0), dog.unsqueeze(0)).item())
print("cat-car:", F.cosine_similarity(cat.unsqueeze(0), car.unsqueeze(0)).item())


cat-dog: 0.234375
cat-car: 0.1328125



Ожидаемый результат: `cat-dog` должно быть заметно выше, чем `cat-car` — оба животные, часто встречаются в похожих контекстах ("питомец", "лапы", "мяукает/лает"), а машина — семантически дальше.

**Важная оговорка для этого конкретного эксперимента**: у многосложных слов (`"cat"` может дать не один токен, а несколько подтокенов в зависимости от контекста и регистра) нужно быть аккуратным с выбором `token_id` — для книги стоит явно проверять через `tokenizer.tokenize(word)`, что слово действительно превращается в один токен, иначе сравнение станет некорректным.

## Токен-эмбеддинги vs "полноценные" text-эмбеддинги — важное разграничение для книги

Это тот момент, где стоит заранее предупредить читателя о путанице, которая часто возникает: **token embedding** (то, что мы только что разобрали) — это статичный вектор для *отдельного токена вне контекста*, полученный простым lookup. Он **не учитывает** окружающие слова — "bank" (берег) и "bank" (банк) на этом этапе получат **один и тот же** вектор, потому что это один и тот же токен ID.

Контекстно-зависимое значение (разное для "bank" в разных предложениях) появляется только **дальше**, после прохождения через слои трансформера с механизмом attention — это уже следующая большая тема (скорее всего, следующий раздел или глава). Явное разграничение "статический lookup-эмбеддинг токена" vs "контекстуальное representation после attention" — один из самых важных концептуальных мостиков всей книги, и стоит сформулировать его чётко именно здесь, до перехода к attention.

## Почему это фундаментально для всего, что мы делали раньше

Хорошая связка назад с предыдущими главами: помнишь, `torch_dtype="auto"` и расчёт памяти GPU (7.6 ГБ для Phi-3)? Embedding-матрица — это **значительная часть** общего числа параметров модели: `32064 × 3072 ≈ 98.5 млн` параметров только на входные эмбеддинги (и часто ещё столько же на выходной слой — если веса не разделяются между input/output embedding, что зависит от архитектуры). Для модели в 3.8 млрд параметров это не главный вклад, но заметный — хороший повод связать абстрактную тему эмбеддингов обратно с конкретными числами памяти, которые ты уже считал раньше.

# Contextualized Word Embeddings From a Language Model (Like BERT) - Clode

Это прямое продолжение того, что мы уже фактически **сделали своими руками** в прошлой серии экспериментов (cat/dog/kitten/car) — только теперь книга формализует это как отдельный раздел и, скорее всего, показывает "канонический" способ через модель-энкодер (обычно именно DeBERTa, о которой мы только что говорили), а не через decoder-модель вроде Phi-3.

## Смысл раздела — то же самое, что мы уже открыли эмпирически

Мы сами только что прошли весь путь: сырые токен-эмбеддинги → contextual embeddings через `output_hidden_states=True`. Этот раздел книги обычно формализует именно это, но чаще на модели-энкодере, потому что:

- **DeBERTa** (и подобные encoder-модели) архитектурно **специализированы** именно под получение качественных contextual representations — в отличие от decoder-only Phi-3, который оптимизирован под *генерацию следующего токена*, а не под representation quality как таковую.
- Encoder видит **весь** текст сразу в обе стороны (bidirectional attention) — токен "bank" в контексте видит и то, что было до, и то, что будет после ("river bank" vs "bank account"). Decoder-модели (как Phi-3) видят только то, что было **до** текущей позиции (causal/masked attention) — это одно из架构ных отличий, которое стоит явно противопоставить.

## Типичный код раздела


In [14]:
from transformers import AutoModel, AutoTokenizer
import torch

model_id = "microsoft/deberta-v3-xsmall"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id)

text = "The bank of the river was muddy after the rain."
tokens = tokenizer(text, return_tensors='pt')

output = model(**tokens)[0]
print(output.shape)  # [1, num_tokens, hidden_size]


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  241MB            

...

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from d

model.safetensors: reconstructing file:   0%|          |  0.00B /  241MB            

model.safetensors: downloading bytes:           |  0.00B            

torch.Size([1, 13, 384])



Обрати внимание: здесь `AutoModel`, а не `AutoModelForCausalLM`, который мы использовали для Phi-3 — потому что нет задачи "предсказать следующий токен", есть задача "получить representation". Соответственно, и `.generate()` тут просто не существует как метод — модель **не умеет** генерировать текст, только кодировать.

## Классический пример раздела — многозначность слова "bank"

Это почти наверняка centerpiece этой части книги, и мы можем прямо здесь его повторить, раз у нас уже отработан паттерн извлечения эмбеддинга конкретного слова из предложения:



In [15]:
def get_word_vector(sentence, word, model, tokenizer):
    tokens = tokenizer(sentence, return_tensors='pt')
    output = model(**tokens)[0][0]  # [seq_len, hidden_size]

    word_ids = tokenizer(word, add_special_tokens=False).input_ids
    full_ids = tokens['input_ids'][0].tolist()

    for i in range(len(full_ids) - len(word_ids) + 1):
        if full_ids[i:i+len(word_ids)] == word_ids:
            return output[i:i+len(word_ids)].mean(dim=0)
    raise ValueError(f"'{word}' не найдено в тексте")


In [16]:
sent1 = "I sat on the bank of the river and watched the water flow."
sent2 = "I went to the bank to withdraw some money."
sent3 = "The dog sat by the river bank."

bank1 = get_word_vector(sent1, "bank", model, tokenizer)
bank2 = get_word_vector(sent2, "bank", model, tokenizer)
bank3 = get_word_vector(sent3, "bank", model, tokenizer)

import torch.nn.functional as F
def cos_sim(a, b):
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

print("bank(река) vs bank(река):", cos_sim(bank1, bank3))
print("bank(река) vs bank(банк):", cos_sim(bank1, bank2))


bank(река) vs bank(река): 0.9306640625
bank(река) vs bank(банк): 0.708984375



**Ожидаемый результат**: "bank" в двух предложениях про реку должны оказаться значительно ближе друг к другу, чем "bank" в значении реки и "bank" в значении финансового учреждения — несмотря на то, что это **буквально одно и то же слово, один и тот же токен ID**. Это прямая, наглядная демонстрация того самого разграничения, которое мы явно проговаривали в конце прошлого раздела: "bank" на уровне сырого input embedding — **всегда один и тот же вектор** (простой lookup по ID), а на уровне contextual embedding после attention — **разные** векторы в зависимости от окружающих слов.

## Чем этот эксперимент лучше нашего предыдущего (cat/dog/kitten)

Наш предыдущий эксперимент сравнивал **разные слова** между собой (cat vs dog vs car) — там всегда есть вопрос "а может это просто разные токены с разными базовыми эмбеддингами, и attention тут ни при чём". Эксперимент с "bank" куда более убедительный **методологически**, потому что сравнивается **один и тот же токен** — единственная переменная, которая меняется, это контекст. Если сходство отличается — это можно объяснить *только* работой attention, никакой альтернативной причины (вроде "разные базовые эмбеддинги токенов") тут просто нет.

Хороший тезис для книги: это классический контролируемый эксперимент — держим всё константным (токен), меняем только одну переменную (контекст) — и наблюдаем эффект.

## Практическое применение — не только иллюстрация, а реальная задача

Дальше раздел обычно связывает это с практикой: contextual word embeddings — основа для задач вроде **Named Entity Recognition** (нужно понять, что "Apple" в "I ate an apple" и "Apple released a new iPhone" — разные сущности, хотя токен один), **word sense disambiguation**, и в целом любой задачи, где значение слова определяется контекстом, а не может быть решено простым словарным lookup.

Если хочешь, могу сразу накидать десяток практических упражнений по этому разделу — благо у нас уже есть готовая инфраструктура (`get_word_vector`, `cos_sim`) и опыт из прошлой серии, так что можно сразу перейти к интересным вариациям: сравнение DeBERTa vs Phi-3 на одной и той же задаче с "bank", проверка на русских омонимах (у тебя это будет особенно ценно для книги — например "лук" в значении растения и оружия), или замер, на каком именно слое трансформера разница между значениями становится максимальной (не обязательно на последнем).

# Contextualized Word Embeddings - Excersizes

## 1. Базовый тест многозначности — "bank" на разных слоях сразу

Начнём с расширения самого показательного эксперимента — посмотрим, как разница между значениями "bank" нарастает **послойно**, а не только на последнем слое:


In [1]:
from transformers import AutoModel, AutoTokenizer
import torch
import torch.nn.functional as F

model_id = "microsoft/deberta-v3-xsmall"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id, output_hidden_states=True)

def get_word_vector_at_layer(sentence, word, layer):
    tokens = tokenizer(sentence, return_tensors='pt')
    with torch.no_grad():
        output = model(**tokens)
    hidden = output.hidden_states[layer][0]

    word_ids = tokenizer(word, add_special_tokens=False).input_ids
    full_ids = tokens['input_ids'][0].tolist()
    for i in range(len(full_ids) - len(word_ids) + 1):
        if full_ids[i:i+len(word_ids)] == word_ids:
            return hidden[i:i+len(word_ids)].mean(dim=0)
    raise ValueError(f"'{word}' не найдено")

def cos_sim(a, b):
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()

sent_river = "I sat on the bank of the river."
sent_money = "I went to the bank to withdraw money."

num_layers = len(model(**tokenizer("test", return_tensors='pt')).hidden_states)
for layer in range(num_layers):
    v1 = get_word_vector_at_layer(sent_river, "bank", layer)
    v2 = get_word_vector_at_layer(sent_money, "bank", layer)
    print(f"Слой {layer}: сходство = {cos_sim(v1, v2):.4f}")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
lm_predictions.lm_head.dense.bias          | UNEXPECTED |  | 
mask_predictions.dense.weight              | UNEXPECTED |  | 
mask_predictions.classifier.bias           | UNEXPECTED |  | 
mask_predictions.dense.bias                | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias            | UNEXPECTED |  | 
lm_predictions.lm_head.bias                | UNEXPECTED |  | 
mask_predictions.classifier.weight         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight          | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED |  | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from d

Слой 0: сходство = 1.0010
Слой 1: сходство = 0.6904
Слой 2: сходство = 0.4683
Слой 3: сходство = 0.3635
Слой 4: сходство = 0.3103
Слой 5: сходство = 0.2651
Слой 6: сходство = 0.2625
Слой 7: сходство = 0.3276
Слой 8: сходство = 0.3398
Слой 9: сходство = 0.4644
Слой 10: сходство = 0.5498
Слой 11: сходство = 0.6729
Слой 12: сходство = 0.7388


**Ожидание**: на слое 0 (это ещё чистый input embedding, до attention) сходство должно быть максимальным (~1.0, ведь это буквально один и тот же токен), а с ростом номера слоя — постепенно снижаться, по мере того как attention "растаскивает" два значения в разные стороны пространства.

## 2. Русский омоним "лук" — твой собственный языковой материал

Раз ты пишешь книгу на русском — замени "bank" на что-то родное аудитории:


In [2]:
sent_plant = "На грядке вырос зелёный лук."
sent_weapon = "Робин Гуд достал лук и стрелы."

v_plant = get_word_vector_at_layer(sent_plant, "лук", -1)
v_weapon = get_word_vector_at_layer(sent_weapon, "лук", -1)

print("Сходство 'лук'(растение) vs 'лук'(оружие):", cos_sim(v_plant, v_weapon))

Сходство 'лук'(растение) vs 'лук'(оружие): 0.96142578125



Хороший контрольный вопрос: справится ли многоязычная DeBERTa/аналог так же хорошо с русской многозначностью, как с английской — это не гарантировано, учитывая то, что мы уже видели про неравномерность языкового покрытия в токенайзерах.

## 3. Контроль — сравнение с однозначным словом

Чтобы доказать, что эффект специфичен именно для омонимов, а не общий шум между разными предложениями:


In [3]:
sent_a = "The cat sat on the mat."
sent_b = "The cat chased the mouse."

v_cat_a = get_word_vector_at_layer(sent_a, "cat", -1)
v_cat_b = get_word_vector_at_layer(sent_b, "cat", -1)

print("Сходство 'cat' в двух разных, но не омонимичных контекстах:", cos_sim(v_cat_a, v_cat_b))

Сходство 'cat' в двух разных, но не омонимичных контекстах: 0.88232421875




**Ожидание**: значение должно быть заметно **выше**, чем для "bank" — "cat" не меняет значения от контекста, лёгкая вариация есть, но не драматический разрыв смысла.

## 4. Три и больше значений одновременно — "spring"

Английское "spring" — редкий случай тройной омонимии, хорошая витрина:


In [4]:
sentences = {
    "сезон": "Flowers bloom in the spring.",
    "пружина": "The mattress has a broken spring inside.",
    "источник": "We drank water from a natural spring.",
}

vectors = {label: get_word_vector_at_layer(text, "spring", -1) for label, text in sentences.items()}

labels = list(vectors.keys())
for i in range(len(labels)):
    for j in range(i+1, len(labels)):
        print(f"{labels[i]} vs {labels[j]}: {cos_sim(vectors[labels[i]], vectors[labels[j]]):.4f}")

сезон vs пружина: 0.7266
сезон vs источник: 0.7764
пружина vs источник: 0.8950




Три попарных сравнения сразу — интересно посмотреть, какая пара значений окажется наименее различима (возможно, "источник" и "сезон" ближе друг к другу, чем к "пружине", просто по общей "природной" теме).

## 5. Влияние позиции слова в предложении на contextual embedding

Проверим, действительно ли **соседние** слова, а не просто "какое-то предложение", определяют вектор:


In [5]:
sent_start = "Bank robbers stole millions today."
sent_end = "Today, robbers stole millions from the bank."

v1 = get_word_vector_at_layer(sent_start, "Bank" if "Bank" in sent_start else "bank", -1)
v2 = get_word_vector_at_layer(sent_end, "bank", -1)

print("Сходство 'bank' в начале vs в конце предложения (тот же смысл):", cos_sim(v1, v2))

Сходство 'bank' в начале vs в конце предложения (тот же смысл): 0.72119140625



Оба предложения про один и тот же финансовый "bank" — ожидание, что сходство будет высоким, несмотря на разную позицию в предложении, разную длину контекста слева/справа. Это проверка того, что модель "поняла" смысл, а не просто зависит от позиции.

## 6. DeBERTa (encoder, bidirectional) vs Phi-3 (decoder, causal) — прямое сравнение архитектур

Самое концептуально важное упражнение раздела — воспроизведём тот же эксперимент на Phi-3 (модель, с которой мы работали весь предыдущий материал) и сравним:



Если хочешь визуализировать это графически (а не просто числами) — можно построить scatter plot через matplotlib, раскрасив точки по метке значения ("река"/"деньги") — отличная иллюстрация для книги, буквально показывающая кластеры в пространстве.

## 10. Практическое применение — простая система разрешения многозначности (WSD) без обучения

Финальное упражнение — соберём всё в законченный мини-инструмент: по новому предложению определяем, какое из известных значений слова имеется в виду, сравнивая с "эталонными" векторами:



Это уже не игрушечный эксперимент, а рабочий (пусть и упрощённый) прототип word sense disambiguation **без всякого fine-tuning** — просто на основе contextual embeddings и косинусного сходства с эталонами. Хороший финал раздела: от "давайте посмотрим на числа" к "вот вам работающий, пусть и примитивный, практический инструмент".

---

**Для книги** — упражнения 1, 6-7 и 9-10, пожалуй, самые сильные кандидаты на включение в основной текст главы: №1 показывает *механику* (послойное нарастание разницы), №6-7 — architectural insight (causal vs bidirectional), а №10 — практическую ценность всей темы, выходящую за рамки "просто интересного эксперимента".

# Contextualized Word Embeddings From a Language Model (Like BERT)

In [8]:
from transformers import AutoModel, AutoTokenizer

# Load a tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

# Load a language model
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

# Tokenize the sentence
tokens = tokenizer('Hello world', return_tensors='pt')

# Process the tokens
output = model(**tokens)[0]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

In [12]:
from transformers import AutoModel, AutoTokenizer

# Load a tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-xsmall")


# Load a language model
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

# Tokenize the sentence
tokens = tokenizer('Hello world', return_tensors='pt')

# Process the tokens
output = model(**tokens)[0]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

In [ ]:
output.shape

torch.Size([1, 4, 384])

In [13]:
for token in tokens['input_ids'][0]:
    print(tokenizer.decode(token))

[CLS]
Hello
world
[SEP]


In [14]:
output

tensor([[[-3.3086, -0.0508, -0.1094,  ..., -0.1703, -0.1619,  0.6919],
         [ 0.8896,  0.0732, -0.1600,  ...,  0.1886,  1.4756,  0.0742],
         [ 0.0859,  0.6357, -0.3040,  ...,  0.4734, -0.1832,  1.0137],
         [-3.1621, -0.1439, -0.0939,  ..., -0.0286, -0.1266,  0.7930]]],
       dtype=torch.float16, grad_fn=<NativeLayerNormBackward0>)

In [15]:
from transformers import AutoTokenizer

tok_v1 = AutoTokenizer.from_pretrained("microsoft/deberta-base")
tok_v3 = AutoTokenizer.from_pretrained("microsoft/deberta-v3-xsmall")

print("deberta-base vocab size:", tok_v1.vocab_size)
print("deberta-v3-xsmall vocab size:", tok_v3.vocab_size)

deberta-base vocab size: 50265
deberta-v3-xsmall vocab size: 128000


Хорошо, что проверил — 50265 vs 128000 наглядно подтверждает, насколько разные словари. Раз токенайзер и модель теперь согласованы (`deberta-v3-xsmall` для обоих), вот набор упражнений именно вокруг этого конкретного кода — детально разбирающих `output = model(**tokens)[0]`.

## 1. Посмотреть на реальную форму выхода и сопоставить с токенами


In [16]:
tokens = tokenizer('Hello world', return_tensors='pt')
output = model(**tokens)[0]

print("Форма выхода:", output.shape)
print("Токены:", tokenizer.convert_ids_to_tokens(tokens['input_ids'][0]))

Форма выхода: torch.Size([1, 4, 384])
Токены: ['[CLS]', '▁Hello', '▁world', '[SEP]']



Обрати внимание — токенов будет **больше двух** (не только "Hello" и "world"), потому что DeBERTa добавляет спецтокены в начало/конец последовательности (аналог `[CLS]`/`[SEP]` у BERT). Первое измерение `output.shape` — это как раз количество этих токенов, включая служебные.

## 2. Что такое `[0]` — сравнение с полным выводом объекта


In [18]:
import torch

full_output = model(**tokens)
print(type(full_output))
print(full_output.keys() if hasattr(full_output, 'keys') else dir(full_output))

# То же самое, что output = model(**tokens)[0]
print(torch.equal(full_output[0], full_output.last_hidden_state))

<class 'transformers.modeling_outputs.BaseModelOutput'>
odict_keys(['last_hidden_state'])
True



Покажет читателю явно, что `[0]` — это не магия, а просто первый элемент структурированного вывода, идентичный `.last_hidden_state`.

## 3. `[CLS]`-токен (или его аналог) как представление всего предложения


In [19]:
cls_vector = output[0][0]  # первый токен последовательности, batch=0
print("Вектор [CLS]:", cls_vector.shape)

Вектор [CLS]: torch.Size([384])



В BERT-семье принято использовать вектор именно первого токена как "summary" всего предложения для задач классификации (потому что через attention он "впитывает" информацию обо всех остальных токенах). Хороший повод объяснить, зачем вообще нужен этот специальный токен в архитектуре.

## 4. Векторы для каждого слова по отдельности


In [20]:
tokens_list = tokenizer.convert_ids_to_tokens(tokens['input_ids'][0])
for i, tok in enumerate(tokens_list):
    vector = output[0][i]
    print(f"{tok:>15}: первые 5 чисел = {vector[:5].tolist()}")

          [CLS]: первые 5 чисел = [-3.30859375, -0.05084228515625, -0.109375, -0.00293731689453125, -0.1436767578125]
         ▁Hello: первые 5 чисел = [0.8896484375, 0.0732421875, -0.1600341796875, 2.181640625, -0.485107421875]
         ▁world: первые 5 чисел = [0.08587646484375, 0.6357421875, -0.303955078125, 1.5400390625, -0.2529296875]
          [SEP]: первые 5 чисел = [-3.162109375, -0.1439208984375, -0.0938720703125, 0.2269287109375, -0.2249755859375]



Наглядная построчная демонстрация — каждый токен получил свой уникальный, контекстуализированный вектор.

## 5. Сравнение с эмбеддингом на входе (до attention) — тот же эксперимент, что делали для Phi-3


In [21]:
input_embeddings = model.get_input_embeddings()
hello_id = tokenizer('Hello', add_special_tokens=False).input_ids[0]

raw_vector = input_embeddings.weight[hello_id]
contextual_vector = output[0][1]  # позиция "Hello" в последовательности (после спецтокена в начале)

import torch.nn.functional as F
similarity = F.cosine_similarity(raw_vector.unsqueeze(0), contextual_vector.unsqueeze(0))
print("Сходство сырого и контекстуализированного эмбеддинга 'Hello':", similarity.item())

Сходство сырого и контекстуализированного эмбеддинга 'Hello': 0.031585693359375




Покажет, насколько сильно attention уже изменил вектор даже для такого короткого, простого предложения.

## 6. Разные предложения — один и тот же токен "world"


In [22]:
def get_output(sentence):
    t = tokenizer(sentence, return_tensors='pt')
    return model(**t)[0][0], tokenizer.convert_ids_to_tokens(t['input_ids'][0])

out1, toks1 = get_output("Hello world")
out2, toks2 = get_output("It's a small world")

idx1 = toks1.index('▁world') if '▁world' in toks1 else [i for i,t in enumerate(toks1) if 'world' in t][0]
idx2 = toks2.index('▁world') if '▁world' in toks2 else [i for i,t in enumerate(toks2) if 'world' in t][0]

vec1 = out1[idx1]
vec2 = out2[idx2]

print(f"{toks1}\n{toks2}")
print("Сходство 'world' в двух контекстах:", F.cosine_similarity(vec1.unsqueeze(0), vec2.unsqueeze(0)).item())


['[CLS]', '▁Hello', '▁world', '[SEP]']
['[CLS]', '▁It', "'", 's', '▁a', '▁small', '▁world', '[SEP]']
Сходство 'world' в двух контекстах: 0.58984375



Хороший переходный мостик к теме многозначности из прошлого раздела — уже на этом простом коде видно, что даже "world" получает разный вектор в зависимости от предложения.

## 7. `attention_mask` — что возвращает токенайзер помимо `input_ids`


In [23]:
print(tokens)
print(tokens.keys())

{'input_ids': tensor([[   1, 5365,  447,    2]]), 'token_type_ids': tensor([[0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1]])}
KeysView({'input_ids': tensor([[   1, 5365,  447,    2]]), 'token_type_ids': tensor([[0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1]])})



Скорее всего увидишь `input_ids` и `attention_mask` (возможно, `token_type_ids`) — хороший повод вернуться к теме attention mask, которую мы обсуждали для Phi-3, но здесь она **обязательна** для корректной работы batching (актуальнее для DeBERTa/BERT-моделей, часто используемых именно в батч-режиме для классификации).

## 8. Батч из нескольких предложений разной длины — паддинг



In [24]:
sentences = ["Hello world", "This is a considerably longer example sentence"]
batch = tokenizer(sentences, return_tensors='pt', padding=True)

print(batch['input_ids'])
print(batch['attention_mask'])

output_batch = model(**batch)[0]
print(output_batch.shape)  # [batch_size, max_seq_len, hidden_size]

tensor([[   1, 5365,  447,    2,    0,    0,    0,    0,    0],
        [   1,  329,  269,  266, 8157, 1032,  738, 4378,    2]])
tensor([[1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1]])
torch.Size([2, 9, 384])



Покажет, как короткое предложение дополняется `[PAD]`-токенами до длины самого длинного в батче, и как `attention_mask` (нули на месте паддинга) не даёт модели "обращать внимание" на эти искусственные токены.

## 9. Сравнение размера модели — DeBERTa-v3-xsmall vs Phi-3


In [25]:
def count_params(m):
    return sum(p.numel() for p in m.parameters())

print(f"DeBERTa-v3-xsmall: {count_params(model)/1e6:.1f}M параметров")
# Для сравнения (если Phi-3 всё ещё загружена в сессии):
# print(f"Phi-3-mini: {count_params(model_phi3)/1e9:.2f}B параметров")

DeBERTa-v3-xsmall: 70.7M параметров



Явный числовой контраст — DeBERTa-v3-xsmall, скорее всего, в районе 22M параметров, против 3.8 **миллиарда** у Phi-3 — разница на три порядка величины. Хороший аргумент в пользу тезиса "для задач классификации не нужна GPT-масштабная модель".

## 10. Скорость инференса — encoder vs decoder на той же задаче


In [26]:
import time

text = "Hello world" * 20

t0 = time.time()
tokens = tokenizer(text, return_tensors='pt')
_ = model(**tokens)
print(f"DeBERTa-v3-xsmall forward pass: {time.time()-t0:.4f}с")

DeBERTa-v3-xsmall forward pass: 1.7131с



Если Phi-3 всё ещё доступна в сессии — сравни с временем одного forward pass через неё на том же тексте. Ожидание: DeBERTa должна быть на порядки быстрее — не только из-за размера, но и потому что здесь **один** проход вперёд, без авторегрессивной генерации токен за токеном (мы подробно разбирали эту разницу ("prompt eval" vs "eval") ещё в главе про Ollama-логи).

---

Упражнения **5-6** здесь особенно ценны — они напрямую продолжают методологию из прошлой большой серии экспериментов (cat/dog/kitten, bank), но теперь на "правильной", специально предназначенной для этого модели-энкодере, а не на decoder-модели, которую мы использовали как компромисс раньше.

# Text Embeddings (For Sentences and Whole Documents)

In [9]:
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# Convert text to text embeddings
vector = model.encode("Best movie ever!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
vector.shape

(768,)

# Word Embeddings Beyond LLMs


In [3]:
!pip install --upgrade gensim


  Using cached gensim-4.4.0-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (8.4 kB)
Using cached gensim-4.4.0-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (27.8 MB)


In [4]:
import gensim.downloader as api

# Download embeddings (66MB, glove, trained on wikipedia, vector size: 50)
# Other options include "word2vec-google-news-300"
# More options at https://github.com/RaRe-Technologies/gensim-data
model = api.load("glove-wiki-gigaword-50")

[==================================================] 100.0% 66.0/66.0MB downloaded


In [5]:
model.most_similar([model['king']], topn=11)

[('king', 1.0000001192092896),
 ('prince', 0.8236179351806641),
 ('queen', 0.7839043140411377),
 ('ii', 0.7746230363845825),
 ('emperor', 0.7736247777938843),
 ('son', 0.766719400882721),
 ('uncle', 0.7627150416374207),
 ('kingdom', 0.7542161345481873),
 ('throne', 0.7539914846420288),
 ('brother', 0.7492411136627197),
 ('ruler', 0.7434253692626953)]

# Recommending songs by embeddings

In [10]:
import pandas as pd
from urllib import request

# Get the playlist dataset file
data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')

# Parse the playlist dataset file. Skip the first two lines as
# they only contain metadata
lines = data.read().decode("utf-8").split('\n')[2:]

# Remove playlists with only one song
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]

# Load song metadata
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns = ['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')

In [7]:
print( 'Playlist #1:\n ', playlists[0], '\n')
print( 'Playlist #2:\n ', playlists[1])

Playlist #1:
  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43'] 

Playlist #2:
  ['78', '79', '80', '3', '62', '81', '14', '82', '48', '83', '84', '17', '85', '86', '87', '88', '74', '89', '90', '91', '4', '73', '62', '92', '17', '53', '59', '93', '94', '51', '50', '27', '95', '48', '96', '97', '98', '99', '100', '57', '101', '102', '25', '103', '3', '104', '105', '106', '107', '47', '108', '109', '110', '111', '112', '113', '25', '63', '62', '114', '115', '84', '116', '117',

In [8]:
from gensim.models import Word2Vec

# Train our Word2Vec model
model = Word2Vec(
    playlists, vector_size=32, window=20, negative=50, min_count=1, workers=4
)

In [9]:
song_id = 2172

# Ask the model for songs similar to song #2172
model.wv.most_similar(positive=str(song_id))

[('5586', 0.9981456398963928),
 ('6658', 0.9976112842559814),
 ('6626', 0.9960243701934814),
 ('5549', 0.9958401322364807),
 ('2849', 0.995378851890564),
 ('6624', 0.9950593113899231),
 ('3167', 0.9946540594100952),
 ('1922', 0.9945700168609619),
 ('2640', 0.9942412972450256),
 ('2704', 0.9941956400871277)]

In [10]:
print(songs_df.iloc[2172])

title     Fade To Black
artist        Metallica
Name: 2172 , dtype: object


In [11]:
import numpy as np

def print_recommendations(song_id):
    similar_songs = np.array(
        model.wv.most_similar(positive=str(song_id),topn=5)
    )[:,0]
    return  songs_df.iloc[similar_songs]

# Extract recommendations
print_recommendations(2172)

,title,artist
id,,
5586,The Last In Line,Dio
6658,(Bang Your Head) Metal Health,Quiet Riot
6626,Blackout,Scorpions
5549,November Rain,Guns N' Roses
2849,Run To The Hills,Iron Maiden


In [12]:
print_recommendations(2172)

,title,artist
id,,
5586,The Last In Line,Dio
6658,(Bang Your Head) Metal Health,Quiet Riot
6626,Blackout,Scorpions
5549,November Rain,Guns N' Roses
2849,Run To The Hills,Iron Maiden


In [13]:
print_recommendations(842)

,title,artist
id,,
6741,Love In This Club (w\/ Young Jeezy),Usher
413,If I Ruled The World (Imagine That) (w\/ Laury...,Nas
5890,Low (w\/ T-Pain),Flo-Rida
890,Knock You Down (w\/ Ne-Yo & Kanye West),Keri Hilson
23316,Eenie Meenie (w\/ Sean Kingston),Justin Bieber
